# Exploración de un archivo `.npz` de representation inputs

Este notebook permite inspeccionar qué contiene un archivo `.npz` generado por `preprocess_representation_inputs.py`.

El objetivo es entender:

- qué arrays se guardan,
- qué forma tienen,
- cuánto pesa el archivo en disco,
- cuánta memoria ocupa al cargarse en RAM,
- qué canales contiene,
- cuál es la frecuencia de muestreo,
- cuánto dura el trial,
- y cómo acceder a una señal específica.

In [11]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


PROJECT_ROOT: Path = Path("/home/russell/ssd/code/Topicos_Ciencia_Datos/Visual_Analytic_DEAP")

NPZ_PATH: Path = (
    PROJECT_ROOT
    / "dataset"
    / "processed"
    / "representation_inputs"
    / "s01"
    / "trial_02_input.npz"
)

print("NPZ path:", NPZ_PATH)
print("Exists:", NPZ_PATH.exists())

NPZ path: /home/russell/ssd/code/Topicos_Ciencia_Datos/Visual_Analytic_DEAP/dataset/processed/representation_inputs/s01/trial_02_input.npz
Exists: True


In [12]:
file_size_bytes: int = NPZ_PATH.stat().st_size
file_size_kb: float = file_size_bytes / 1024
file_size_mb: float = file_size_kb / 1024

print(f"File size: {file_size_bytes:,} bytes")
print(f"File size: {file_size_kb:.2f} KB")
print(f"File size: {file_size_mb:.4f} MB")

File size: 2,528,183 bytes
File size: 2468.93 KB
File size: 2.4111 MB


In [13]:
npz_data: Any = np.load(NPZ_PATH, allow_pickle=True)

print("Keys inside .npz:")
for key in npz_data.files:
    print("-", key)

Keys inside .npz:
- signals
- times
- channels
- sfreq
- participant_id
- trial
- experiment_id


In [14]:
signals: np.ndarray = np.asarray(npz_data["signals"])
times: np.ndarray = np.asarray(npz_data["times"])
channels: list[str] = [str(channel) for channel in npz_data["channels"].tolist()]
sfreq: float = float(npz_data["sfreq"][0])
participant_id: int = int(npz_data["participant_id"][0])
trial: int = int(npz_data["trial"][0])
experiment_id: int = int(npz_data["experiment_id"][0])

print("participant_id:", participant_id)
print("trial:", trial)
print("experiment_id:", experiment_id)
print("sfreq:", sfreq)
print("signals shape:", signals.shape)
print("times shape:", times.shape)
print("num channels:", len(channels))

participant_id: 1
trial: 2
experiment_id: 18
sfreq: 128.0
signals shape: (44, 7680)
times shape: (7680,)
num channels: 44


In [15]:
num_channels: int = signals.shape[0]
num_samples: int = signals.shape[1]
duration_sec: float = num_samples / sfreq

print(f"Signals shape = ({num_channels}, {num_samples})")
print(f"Duration = {duration_sec:.4f} seconds")
print(f"Expected samples for 60s = {int(60 * sfreq)}")

Signals shape = (44, 7680)
Duration = 60.0000 seconds
Expected samples for 60s = 7680


In [16]:
signals_ram_bytes: int = signals.nbytes
times_ram_bytes: int = times.nbytes
total_ram_bytes: int = signals_ram_bytes + times_ram_bytes

print(f"signals RAM: {signals_ram_bytes:,} bytes = {signals_ram_bytes / 1024**2:.4f} MB")
print(f"times RAM:   {times_ram_bytes:,} bytes = {times_ram_bytes / 1024**2:.4f} MB")
print(f"total RAM:   {total_ram_bytes:,} bytes = {total_ram_bytes / 1024**2:.4f} MB")
print("signals dtype:", signals.dtype)
print("times dtype:", times.dtype)

signals RAM: 2,703,360 bytes = 2.5781 MB
times RAM:   61,440 bytes = 0.0586 MB
total RAM:   2,764,800 bytes = 2.6367 MB
signals dtype: float64
times dtype: float64


In [17]:
channels_df: pd.DataFrame = pd.DataFrame(
    {
        "index": list(range(len(channels))),
        "channel": channels,
    }
)

display(channels_df)

,index,channel
0,0,Fp1
1,1,AF3
2,2,F3
3,3,F7
4,4,FC5
5,5,FC1
6,6,C3
7,7,T7
8,8,CP5
9,9,CP1


In [18]:
eeg_channels: list[str] = channels[:32]
eog_channels: list[str] = channels[32:36]
emg_channels: list[str] = channels[36:40]
peripheral_channels: list[str] = channels[40:44]

print("EEG:", eeg_channels)
print("EOG:", eog_channels)
print("EMG:", emg_channels)
print("Peripheral:", peripheral_channels)

EEG: ['Fp1', 'AF3', 'F3', 'F7', 'FC5', 'FC1', 'C3', 'T7', 'CP5', 'CP1', 'P3', 'P7', 'PO3', 'O1', 'Oz', 'Pz', 'Fp2', 'AF4', 'Fz', 'F4', 'F8', 'FC6', 'FC2', 'Cz', 'C4', 'T8', 'CP6', 'CP2', 'P4', 'P8', 'PO4', 'O2']
EOG: ['EXG1', 'EXG2', 'EXG3', 'EXG4']
EMG: ['EXG5', 'EXG6', 'EXG7', 'EXG8']
Peripheral: ['GSR1', 'Resp', 'Plet', 'Temp']


In [19]:
summary_rows: list[dict[str, float | str]] = []

for channel_index, channel_name in enumerate(channels):
    values: np.ndarray = signals[channel_index]

    summary_rows.append(
        {
            "channel": channel_name,
            "mean": float(np.mean(values)),
            "std": float(np.std(values)),
            "min": float(np.min(values)),
            "max": float(np.max(values)),
            "rms": float(np.sqrt(np.mean(values ** 2))),
        }
    )

summary_df: pd.DataFrame = pd.DataFrame(summary_rows)

display(summary_df)

,channel,mean,std,min,max,rms
0,Fp1,-1.815598e-08,0.000007,-0.000025,0.000020,0.000007
1,AF3,-1.835524e-08,0.000007,-0.000024,0.000023,0.000007
2,F3,-1.742491e-08,0.000007,-0.000024,0.000020,0.000007
3,F7,-2.036123e-08,0.000008,-0.000030,0.000032,0.000008
4,FC5,-1.888778e-08,0.000007,-0.000024,0.000024,0.000007
5,FC1,-1.446003e-08,0.000006,-0.000023,0.000017,0.000006
6,C3,-1.478862e-08,0.000006,-0.000022,0.000015,0.000006
7,T7,-2.073932e-08,0.000008,-0.000029,0.000033,0.000008
8,CP5,-1.414795e-08,0.000007,-0.000023,0.000021,0.000007
9,CP1,-1.123402e-08,0.000006,-0.000022,0.000017,0.000006


In [20]:
channel_name: str = "Fp1"

channel_index: int = channels.index(channel_name)
channel_signal: np.ndarray = signals[channel_index]

print("Channel:", channel_name)
print("Index:", channel_index)
print("Shape:", channel_signal.shape)
print("First 10 values:")
print(channel_signal[:10])

Channel: Fp1
Index: 0
Shape: (7680,)
First 10 values:
[ 8.02442103e-09 -1.96812057e-05 -6.34759910e-06 -1.46652997e-05
 -2.40551173e-05 -1.03949152e-05 -1.99354265e-05 -9.15086128e-06
  3.70645823e-06 -9.62015701e-06]


## Conclusión

Este archivo `.npz` representa un solo trial procesado para la etapa de representaciones.

En este caso:

- cada archivo corresponde a un participante y trial específico,
- `signals` contiene las señales multicanal durante la fase `During`,
- `times` contiene el eje temporal relativo,
- `channels` conserva el orden de los canales,
- `sfreq` indica la frecuencia procesada,
- y la metadata permite relacionar el trial con su participante y estímulo.

Estos archivos serán la entrada común para:

1. generación de vectores de características basados en el paper DEAP,
2. generación futura de embeddings aprendidos con deep learning.